# M-Detector Debugging and Tuning Notebook
## 0. Setup and Imports

In [ ]:
RUN_K3D = False

In [ ]:
# %%
import os
import sys
# import yaml # yaml is now handled by MDetectorConfigAccessor
import numpy as np
import matplotlib.pyplot as plt
import h5py
from tqdm.notebook import tqdm 
from pyquaternion import Quaternion
import logging
from typing import Dict, List, Optional, Tuple, Any
import k3d
import json

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('') 
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules (after adding project root) ---
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box as NuScenesDataClassesBox

# --- MODIFIED: Import MDetectorConfigAccessor ---
from src.config_loader import MDetectorConfigAccessor

# Core M-Detector components
from src.core.m_detector.base import MDetector
from src.core.depth_image import DepthImage
from src.core.constants import OcclusionResult, POINT_LABEL_DTYPE
from src.core.debug_collector import PointDebugCollector

# Data utilities
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence, get_lidar_sweep_data
from src.utils.transformations import transform_points_numpy
from src.utils.validation_utils import get_gt_dynamic_points_for_sweep 
from src.utils.notebook_helpers import (
    load_gt_labels_for_sweep_from_hdf5, 
    load_mdet_results_for_sweep_from_hdf5,
    create_k3d_plot_with_points 
)
# Debug helpers
from src.utils.debug_helpers import debug_point_m_detector_logic, get_misclassified_points
# Visualization specific for this notebook if needed
from src.visualization.debug_animation_helpers import create_point_debug_bev_animation
from src.visualization.k3d_visualizer import visualize_sweep_k3d
from src.data_utils.label_generation import find_instances_in_scene, get_interpolated_extrapolated_boxes_for_instance 

print("Cell 0: Imports and Paths - OK")

In [ ]:
# %% Cell 1: Logging, Configuration, NuScenes Init, Paths, Scene/Sweep Selection

# --- Configure Logging for M-Detector ---
# (Logging setup remains the same - this is good)
mdetector_logger = logging.getLogger('src.core.m_detector.base')
mdetector_logger.setLevel(logging.INFO) # Set to INFO for less verbose, DEBUG for more
processing_logger = logging.getLogger('src.core.m_detector.processing')
processing_logger.setLevel(logging.INFO)
map_consistency_logger = logging.getLogger('src.core.m_detector.map_consistency') # Uncomment if used
map_consistency_logger.setLevel(logging.INFO)
# Add a stream handler to see logs in the notebook output
# stream_handler = logging.StreamHandler(sys.stdout)
# formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
# stream_handler.setFormatter(formatter)
# if not mdetector_logger.hasHandlers(): mdetector_logger.addHandler(stream_handler)
# if not processing_logger.hasHandlers(): processing_logger.addHandler(stream_handler)
# if not map_consistency_logger.hasHandlers(): map_consistency_logger.addHandler(stream_handler)
print("Logging configured (set to INFO for key modules, adjust as needed).")


# --- Load Configuration using Accessor --- # MODIFIED
config_accessor: Optional[MDetectorConfigAccessor] = None
try:
    config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    config_accessor = MDetectorConfigAccessor(config_file_path)
    print(f"Configuration loaded successfully using MDetectorConfigAccessor from: {config_file_path}")
except FileNotFoundError:
    print(f"ERROR: Config file not found at {config_file_path}. Please check the path.")
except Exception as e:
    print(f"Error loading config with MDetectorConfigAccessor: {e}")

# --- Initialize NuScenes ---
nusc: Optional[NuScenes] = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    try:
        nusc = NuScenes(
            version=nuscenes_params.get('version'),
            dataroot=nuscenes_params.get('dataroot'),
            verbose=nuscenes_params.get('verbose_init', False) # Use verbose_init from config
        )
        print(f"NuScenes SDK initialized (Version: {nuscenes_params.get('version')}).")
    except Exception as e:
        print(f"Error initializing NuScenes SDK: {e}. Check 'version' and 'dataroot'.")
else:
    print("ERROR: MDetectorConfigAccessor failed. Cannot initialize NuScenes SDK.")

# --- Define Paths for GT Labels and M-Detector Outputs (HDF5) ---
GT_LABELS_HDF5_DIR: Optional[str] = None
MDET_RESULTS_HDF5_DIR: Optional[str] = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    GT_LABELS_HDF5_DIR = nuscenes_params.get('label_path') # Should point to dir with gt_point_labels_SCENENAME.h5
    if GT_LABELS_HDF5_DIR and not os.path.isabs(GT_LABELS_HDF5_DIR):
        GT_LABELS_HDF5_DIR = os.path.join(PROJECT_ROOT, GT_LABELS_HDF5_DIR)

    mdet_output_paths = config_accessor.get_mdetector_output_paths()
    MDET_RESULTS_HDF5_DIR = mdet_output_paths.get('save_path') # Dir where mdet_results_SCENENAME.h5 are saved
    if MDET_RESULTS_HDF5_DIR and not os.path.isabs(MDET_RESULTS_HDF5_DIR):
        MDET_RESULTS_HDF5_DIR = os.path.join(PROJECT_ROOT, MDET_RESULTS_HDF5_DIR)

print(f"GT Labels HDF5 Directory: {GT_LABELS_HDF5_DIR if GT_LABELS_HDF5_DIR else 'Not configured'}")
print(f"M-Detector Results HDF5 Directory: {MDET_RESULTS_HDF5_DIR if MDET_RESULTS_HDF5_DIR else 'Not configured'}")


# --- Scene and Sweep Selection ---
TARGET_SCENE_NAME = 'scene-0103' # Keep as is, or make it configurable via accessor if desired for notebook
# Example:
# if config_accessor:
#   debug_params = config_accessor.get_debug_settings() # Assuming a new section
#   TARGET_SCENE_NAME = debug_params.get('target_scene_for_notebook', 'scene-0103')

# These are specific to this notebook's debugging flow, not general config
SKIP_SWEEPS_MANUAL = 40 # Number of initial sweeps to feed to MDet before the target sweep
MAX_SWEEPS_TO_PRIME = 20 # Max sweeps to process *before* the target one for priming history
TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP = 10 # The index of our *target* sweep, relative to the start of the scene *after* SKIP_SWEEPS_MANUAL

# Derived values
target_scene_rec: Optional[Dict[str, Any]] = None
TARGET_LIDAR_SD_TOKEN: Optional[str] = None
TARGET_SWEEP_DATA_DICT: Optional[Dict[str, Any]] = None
scene_sweeps_full_sequence: List[Dict[str, Any]] = [] # All sweeps to feed to MDetector up to and including target

if nusc and config_accessor:
    # Find the target scene record
    scene_found = False
    for scene in nusc.scene:
        if scene['name'] == TARGET_SCENE_NAME:
            target_scene_rec = scene
            scene_found = True
            break
    if not scene_found:
        print(f"ERROR: Scene '{TARGET_SCENE_NAME}' not found in NuScenes dataset.")
        target_scene_rec = None # Ensure it's None
    
    if target_scene_rec:
        print(f"Target Scene: '{target_scene_rec['name']}' (Token: {target_scene_rec['token']})")
        
        # Get all sweeps for the scene
        all_sweeps_for_scene = list(get_scene_sweep_data_sequence(nusc, target_scene_rec['token']))
        
        # Determine the actual index of the target sweep in the full list of scene sweeps
        # The logic is: we skip some, then process some for priming, then hit our target.
        # For this notebook, let's simplify: we will feed sweeps up to a certain point.
        # The TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP is the index *within* the sweeps we choose to *consider* for this debug run.
        
        # Let's define the sequence of sweeps to feed to MDetector for this debug session
        # It will include sweeps for priming and the target sweep itself.
        
        # Example: if SKIP_SWEEPS_MANUAL = 40, and TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP = 10
        # This means the target sweep is the (40+10) = 50th sweep in the scene (0-indexed).
        # We will feed sweeps from index 40 up to 50 (inclusive).
        
        start_feed_idx = SKIP_SWEEPS_MANUAL
        # The target sweep is the 10th sweep *after* the skipped ones.
        # So, its index in all_sweeps_for_scene is start_feed_idx + TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP
        target_sweep_actual_idx_in_scene = start_feed_idx + TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP

        if 0 <= target_sweep_actual_idx_in_scene < len(all_sweeps_for_scene):
            # Define the sequence of sweeps to feed to MDetector for this debug session
            # This includes the sweeps for priming (up to MAX_SWEEPS_TO_PRIME before target)
            # and the target sweep itself.
            
            # Determine the first sweep to actually process for priming before the target
            first_prime_sweep_idx = max(start_feed_idx, target_sweep_actual_idx_in_scene - MAX_SWEEPS_TO_PRIME)
            
            # The sequence to feed goes from first_prime_sweep_idx up to target_sweep_actual_idx_in_scene (inclusive)
            scene_sweeps_full_sequence = all_sweeps_for_scene[first_prime_sweep_idx : target_sweep_actual_idx_in_scene + 1]

            if scene_sweeps_full_sequence:
                TARGET_SWEEP_DATA_DICT = scene_sweeps_full_sequence[-1] # The last one is our target
                TARGET_LIDAR_SD_TOKEN = TARGET_SWEEP_DATA_DICT['lidar_sd_token']
                print(f"  Selected TARGET_LIDAR_SD_TOKEN: ...{TARGET_LIDAR_SD_TOKEN[-8:]}")
                print(f"  This is sweep index {target_sweep_actual_idx_in_scene} in the full scene.")
                print(f"  M-Detector will be fed {len(scene_sweeps_full_sequence)} sweeps for this debug run (from index {first_prime_sweep_idx} to {target_sweep_actual_idx_in_scene} of scene).")
            else:
                print(f"ERROR: Could not form a sequence of sweeps to feed. Check SKIP_SWEEPS_MANUAL, MAX_SWEEPS_TO_PRIME, and TARGET_SWEEP_INDEX_IN_SCENE_AFTER_SKIP.")
        else:
            print(f"ERROR: Calculated target_sweep_actual_idx_in_scene ({target_sweep_actual_idx_in_scene}) is out of bounds for scene '{TARGET_SCENE_NAME}' (has {len(all_sweeps_for_scene)} sweeps).")
            
else:
    print("Skipping Scene/Sweep Selection: NuScenes SDK not initialized or Config Accessor failed.")

print("\nCell 1: Setup and Configuration - OK")

In [ ]:
# %% Cell 2: M-Detector Initialization

detector: Optional[MDetector] = None
processed_target_di: Optional[DepthImage] = None # Will hold the target DI after being added to MDet
idx_of_processed_target_di: Optional[int] = None   # Index of target DI in MDet library

if config_accessor:
    try:
        print("\nInitializing M-Detector...")
        # --- MODIFIED: Pass config_accessor to MDetector ---
        detector = MDetector(config_accessor=config_accessor)
        # Optionally, set a debug collector if you want to trace all points during the priming phase
        # This is different from the collector set in Cell 5 for specific point debugging.
        # detector.set_debug_collector(PointDebugCollector(points_to_trace=None)) # Example: trace all
        print("M-Detector initialized successfully.")
    except Exception as e:
        print(f"Error initializing M-Detector: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Skipping M-Detector initialization: Config Accessor not available.")

print("\nCell 2: M-Detector Initialization - OK")

In [ ]:
# %% Cell 3: M-Detector Library Priming & Target DI Preparation

if detector and target_scene_rec and scene_sweeps_full_sequence and TARGET_LIDAR_SD_TOKEN:
    print(f"\nProcessing sweeps for scene '{target_scene_rec['name']}' to prime M-Detector library...")
    
    detector.reset_scene_state() 
    
    for i, sweep_data in enumerate(tqdm(scene_sweeps_full_sequence, desc="Feeding Sweeps to M-Detector")):
        added_di = detector.add_sweep_and_create_depth_image(
            points_lidar_frame=sweep_data['points_sensor_frame'],
            T_global_lidar=sweep_data['T_global_lidar'],
            lidar_timestamp=sweep_data['timestamp'],
            lidar_sd_token=sweep_data['lidar_sd_token']
        )
        
        if not added_di: # Check if DI creation failed
            print(f"Warning: Failed to create DepthImage for sweep {sweep_data['lidar_sd_token']}. Skipping this sweep.")
            if sweep_data['lidar_sd_token'] == TARGET_LIDAR_SD_TOKEN:
                print(f"ERROR: Target sweep {TARGET_LIDAR_SD_TOKEN} could not be added as a DepthImage.")
                processed_target_di = None # Ensure it's None
                break
            continue

        if sweep_data['lidar_sd_token'] == TARGET_LIDAR_SD_TOKEN:
            processed_target_di = added_di
            try:
                # Find its index in the library's current state
                # Ensure library is not empty before trying to get index
                if detector.depth_image_library and len(detector.depth_image_library.get_all_images()) > 0:
                    idx_of_processed_target_di = detector.depth_image_library.get_all_images().index(processed_target_di)
                else: # Should not happen if added_di was successful
                    idx_of_processed_target_di = -1
            except ValueError:
                print(f"ERROR: Target DI (Token: {TARGET_LIDAR_SD_TOKEN}) was added but not found in library. This is unexpected.")
                idx_of_processed_target_di = -1 
            
            if idx_of_processed_target_di != -1:
                print(f"\nTarget sweep {TARGET_LIDAR_SD_TOKEN[:8]}... added to M-Detector library at index {idx_of_processed_target_di}.")
                print(f"  It is now ready for manual processing and inspection.")
            break 
        
        else: 
            detector.decide_and_process_frame(is_end_of_sequence=False) # is_end_of_sequence is False during priming


    if not processed_target_di:
        print(f"ERROR: Target sweep {TARGET_LIDAR_SD_TOKEN} was not reached or processed correctly in the loop.")
    elif idx_of_processed_target_di is None or idx_of_processed_target_di == -1: # Check if None explicitly
        print(f"ERROR: Target DI was added but its index in library could not be confirmed.")

else:
    print("Skipping M-Detector library priming: M-Detector not instantiated, scene/sweep not defined, or no sweeps in sequence.")

print("\nCell 3: M-Detector Library Priming - OK")

In [ ]:
# %% Cell 4: Ground Truth Label Loading for Target DI

gt_indices_dict_for_target: Optional[Dict[str, Any]] = None # Changed type hint for Any
velocity_threshold_gt: Optional[float] = None # Initialize

if nusc and TARGET_SWEEP_DATA_DICT and processed_target_di and \
   processed_target_di.original_points_global_coords is not None and \
   target_scene_rec and GT_LABELS_HDF5_DIR and config_accessor and detector: # MODIFIED: Added config_accessor
    
    print(f"\nLoading Ground Truth labels for the target sweep (Token: {TARGET_LIDAR_SD_TOKEN[:8]}...).")
    
    gt_labels_scene_hdf5_filepath = os.path.join(GT_LABELS_HDF5_DIR, f"gt_point_labels_{target_scene_rec['name']}.h5")
    
    # --- MODIFIED: Fetch parameters using config_accessor ---
    validation_params = config_accessor.get_validation_params()
    velocity_threshold_gt = validation_params.get('gt_velocity_threshold', 0.5) # Default if not in config

    # Get M-Detector's filtering params from its config_accessor
    # These are the ranges M-Detector *would have used* for this run.
    point_pre_filtering_params = config_accessor.get_point_pre_filtering_params()
    mdet_min_range = point_pre_filtering_params.get('min_range_meters', 1.0)
    mdet_max_range = point_pre_filtering_params.get('max_range_meters', 80.0)
    # --- End MODIFIED ---
    
    print(f"  Using GT velocity threshold: {velocity_threshold_gt:.2f} m/s")
    print(f"  Using M-Detector's range filter for GT consistency: min={mdet_min_range:.1f}m, max={mdet_max_range:.1f}m")

    if os.path.exists(gt_labels_scene_hdf5_filepath):
        gt_indices_dict_for_target = get_gt_dynamic_points_for_sweep(
            nusc=nusc,
            sweep_data_dict=TARGET_SWEEP_DATA_DICT, 
            all_points_global_mdetector=processed_target_di.original_points_global_coords,
            gt_labels_scene_hdf5_path=gt_labels_scene_hdf5_filepath,
            velocity_threshold=velocity_threshold_gt,
            mdetector_min_range=mdet_min_range, 
            mdetector_max_range=mdet_max_range
        )
        
        if gt_indices_dict_for_target: # Check if dict is not None
            print(f"  [Returned from get_gt_dynamic_points_for_sweep DEBUG]:")
            print(f"    Raw GT HDF5 points for sweep: {gt_indices_dict_for_target.get('debug_gt_hdf5_raw_count', 'N/A')}")
            print(f"    Filtered GT HDF5 points (using MDet range): {gt_indices_dict_for_target.get('debug_gt_hdf5_filtered_count', 'N/A')}")
            print(f"    M-Detector input points for sweep: {gt_indices_dict_for_target.get('debug_mdet_input_count', 'N/A')}")

            if gt_indices_dict_for_target.get('error_msg') is None:
                print("\nGround Truth INDICES loaded and categorized for the target sweep (index-based).")
                print(f"  GT Dynamic point INDICES found: {len(gt_indices_dict_for_target.get('gt_dynamic_indices', []))}")
                print(f"  GT Static point INDICES found: {len(gt_indices_dict_for_target.get('gt_static_indices', []))}")
                print(f"  GT Unlabeled point INDICES found: {len(gt_indices_dict_for_target.get('unlabeled_indices', []))}")
            else:
                print(f"\nFailed to get GT indices for target sweep. Error: {gt_indices_dict_for_target.get('error_msg')}")
        else:
            print(f"\nFunction get_gt_dynamic_points_for_sweep returned None (unexpected).")
    else:
        print(f"ERROR: GT HDF5 file not found: {gt_labels_scene_hdf5_filepath}")
else:
    print("Skipping Ground Truth loading: Prerequisites not met (nusc, target sweep data, processed_target_di, GT path, config_accessor, or detector missing).")

print("\nCell 4: GT Label Loading - OK")

In [ ]:
# %% Cell 5: Process Target DI & Identify Misclassifications

misclassified_summary: Optional[Dict[str, List[int]]] = None
debug_collector: Optional[PointDebugCollector] = None # Initialize

if detector and processed_target_di and idx_of_processed_target_di is not None and idx_of_processed_target_di != -1:
    print(f"\nManually processing target DI: TS {processed_target_di.timestamp} (Index in lib: {idx_of_processed_target_di})")
    
    debug_collector = PointDebugCollector(points_to_trace=[]) 
    detector.set_debug_collector(debug_collector)

    result_dict = detector.process_and_label_di(processed_target_di, idx_of_processed_target_di)
    
    if result_dict.get('success'):
        print(f"Processing successful for target DI.")
        print(f"  Points labeled: {result_dict.get('points_labeled')}")
        label_counts_from_result = result_dict.get('label_counts', {})
        if label_counts_from_result:
            formatted_label_counts = {}
            for k_val, v_count in label_counts_from_result.items():
                if isinstance(k_val, OcclusionResult):
                    formatted_label_counts[k_val.name] = v_count
                elif isinstance(k_val, int):
                    try:
                        formatted_label_counts[OcclusionResult(k_val).name] = v_count
                    except ValueError: # Handle if int is not a valid OcclusionResult value
                        formatted_label_counts[f"UnknownLabelValue({k_val})"] = v_count
                else: # Should not happen if keys are OcclusionResult or int
                    formatted_label_counts[str(k_val)] = v_count 
            print(f"  Label counts: {formatted_label_counts}")
        else:
            print("  Label counts not available in result_dict.")
            
        if gt_indices_dict_for_target and gt_indices_dict_for_target.get('error_msg') is None:
            print("\nIdentifying misclassifications...")
            if hasattr(processed_target_di, 'mdet_labels_for_points') and processed_target_di.mdet_labels_for_points is not None:
                misclassified_summary = get_misclassified_points(processed_target_di, gt_indices_dict_for_target)
                if misclassified_summary:
                    fp_to_debug = misclassified_summary.get('fp_dynamic', [])
                    fn_to_debug = misclassified_summary.get('fn_dynamic', [])
                    if fp_to_debug:
                        print(f"  Sample False Positives (Static/Unlabeled -> MDet Dynamic): {fp_to_debug[:min(5, len(fp_to_debug))]}")
                    if fn_to_debug:
                        print(f"  Sample False Negatives (Dynamic -> MDet Not Dynamic): {fn_to_debug[:min(5, len(fn_to_debug))]}")
            else:
                print("  M-Detector labels not found on processed_target_di. Cannot identify misclassifications.")
        else:
            print("Ground Truth not available or had errors for target sweep, cannot identify misclassifications.")
    else:
        print(f"Processing FAILED for target DI. Reason: {result_dict.get('reason')}")
else:
    print("Cannot process target DI: M-Detector not ready, or target DI not identified/found in library.")

print("\nCell 5: Target DI Processing - OK")

In [ ]:
# %% Cell 6: Deep Dive into Specific Misclassified/Interesting Points - GT Info

# --- User Input: Select a point index to debug ---
pt_idx_to_debug: Optional[int] = None # Initialize

if 'misclassified_summary' in locals() and misclassified_summary:
    fp_dynamic_list = misclassified_summary.get('fp_dynamic', [])
    fn_dynamic_list = misclassified_summary.get('fn_dynamic', [])
    if fp_dynamic_list:
        pt_idx_to_debug = fp_dynamic_list[0] 
        print(f"Selected pt_idx_to_debug from False Positives: {pt_idx_to_debug}")
    elif fn_dynamic_list:
        pt_idx_to_debug = fn_dynamic_list[0]
        print(f"No False Positives found. Selected pt_idx_to_debug from False Negatives: {pt_idx_to_debug}")
    else:
        pt_idx_to_debug = 100 # Fallback
        print(f"No specific misclassifications to pick from. Using default pt_idx_to_debug: {pt_idx_to_debug}")
else:
    pt_idx_to_debug = 100 # Fallback if misclassified_summary is not available
    print(f"Misclassified summary not available. Using default pt_idx_to_debug: {pt_idx_to_debug}")

# Ensure velocity_threshold_gt was set in Cell 4
if 'velocity_threshold_gt' not in locals() or velocity_threshold_gt is None:
    if config_accessor:
        validation_params = config_accessor.get_validation_params()
        velocity_threshold_gt = validation_params.get('gt_velocity_threshold', 0.5)
        print(f"  (velocity_threshold_gt was not set, fetched from config: {velocity_threshold_gt})")
    else:
        velocity_threshold_gt = 0.5 # Absolute fallback
        print(f"  (velocity_threshold_gt was not set and config_accessor missing, using default: {velocity_threshold_gt})")


if gt_indices_dict_for_target and gt_indices_dict_for_target.get('error_msg') is None:
    all_gt_labels_array = gt_indices_dict_for_target.get('all_gt_point_labels_filtered')
    if all_gt_labels_array is not None and pt_idx_to_debug is not None:
        if 0 <= pt_idx_to_debug < len(all_gt_labels_array):
            gt_info_for_debug_pt = all_gt_labels_array[pt_idx_to_debug]
            gt_vx = gt_info_for_debug_pt['velocity_x']
            gt_vy = gt_info_for_debug_pt['velocity_y']
            gt_speed = np.sqrt(gt_vx**2 + gt_vy**2)
            gt_instance_token = gt_info_for_debug_pt['instance_token'].decode('utf-8', 'ignore') if gt_info_for_debug_pt['instance_token'] else "N/A"
            
            print(f"\n  Ground Truth Info for P{pt_idx_to_debug}:")
            print(f"    GT Velocity (vx, vy): ({gt_vx:.2f}, {gt_vy:.2f}) m/s")
            print(f"    GT Speed: {gt_speed:.2f} m/s (Threshold for dynamic: {velocity_threshold_gt} m/s)") # Use defined var
            print(f"    GT Instance Token: {gt_instance_token}")
            if gt_speed < velocity_threshold_gt:
                print(f"    ---> P{pt_idx_to_debug} is GT STATIC (speed {gt_speed:.2f} < {velocity_threshold_gt:.2f})")
            else:
                print(f"    ---> P{pt_idx_to_debug} is GT DYNAMIC (speed {gt_speed:.2f} >= {velocity_threshold_gt:.2f})")
        else:
            print(f"  Warning: pt_idx_to_debug {pt_idx_to_debug} is out of bounds for the loaded GT labels array (len {len(all_gt_labels_array if all_gt_labels_array is not None else 0)}).")
    elif gt_indices_dict_for_target and gt_indices_dict_for_target.get('error_msg'):
        print(f"  Could not get detailed GT info for P{pt_idx_to_debug} due to earlier error: {gt_indices_dict_for_target.get('error_msg')}")
    else: # all_gt_labels_array is None
        print(f"  'all_gt_point_labels_filtered' not found in gt_indices_dict_for_target. Cannot get GT info for P{pt_idx_to_debug}.")
else:
    print(f"  GT indices dictionary not available or had errors. Cannot get GT info for P{pt_idx_to_debug}.")


# Raw check remains the same
if nusc and TARGET_LIDAR_SD_TOKEN and pt_idx_to_debug is not None:
    raw_target_points_sf, raw_target_T_global_lidar, _, _, _, _, _ = \
        get_lidar_sweep_data(nusc, TARGET_LIDAR_SD_TOKEN)
    if raw_target_points_sf.shape[0] > 0 :
        raw_target_points_global = transform_points_numpy(raw_target_points_sf, raw_target_T_global_lidar)
        if raw_target_points_global.shape[0] > pt_idx_to_debug:
            raw_global_coord_of_debug_pt = raw_target_points_global[pt_idx_to_debug, :3]
            print(f"RAW CHECK: Global coord of pt {pt_idx_to_debug} from fresh load of {TARGET_LIDAR_SD_TOKEN[:8]}: {np.round(raw_global_coord_of_debug_pt, 7).tolist()}")
        else:
            print(f"RAW CHECK: pt_idx {pt_idx_to_debug} out of bounds for fresh load ({raw_target_points_global.shape[0]} points).")
    else:
        print(f"RAW CHECK: No points in fresh load of {TARGET_LIDAR_SD_TOKEN[:8]}.")


print("\nCell 6: Deep Dive GT Info - OK")

In [ ]:
# %% Cell 7: Deep Dive - M-Detector Logic Trace

# Ensure debug_collector from Cell 5 is used or re-initialized if needed
if 'debug_collector' not in locals() or debug_collector is None:
    debug_collector = PointDebugCollector(points_to_trace=[]) # Default if not created in Cell 5
    if detector:
        detector.set_debug_collector(debug_collector)

if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   processed_target_di and processed_target_di.original_points_global_coords is not None and \
   idx_of_processed_target_di is not None and idx_of_processed_target_di != -1 and \
   detector:

    if pt_idx_to_debug >= len(processed_target_di.original_points_global_coords):
        print(f"ERROR: pt_idx_to_debug ({pt_idx_to_debug}) is out of bounds for processed_target_di.")
    else:
        # Set the specific point to trace for this deep dive in the collector
        if debug_collector:
            debug_collector.set_points_to_trace([pt_idx_to_debug])
            print(f"DebugCollector set to trace point: {pt_idx_to_debug}")
        
        print(f"\nInitiating M-Detector logic deep dive for Point Index: {pt_idx_to_debug}")
        # debug_point_m_detector_logic was updated to use detector_instance.config_accessor
        debug_point_m_detector_logic(
            pt_idx=pt_idx_to_debug,
            current_di=processed_target_di,
            current_di_idx_in_lib=idx_of_processed_target_di,
            detector_instance=detector,
            gt_indices_dict=gt_indices_dict_for_target,
            debug_collector_instance=debug_collector
        )
else:
    print("Cannot perform M-Detector logic deep dive: Prerequisites not met.")


In [ ]:

# %% Cell 8: Debug Animation for Specific Point

if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   detector is not None and target_scene_rec is not None and \
   processed_target_di is not None and idx_of_processed_target_di is not None and \
   TARGET_LIDAR_SD_TOKEN is not None and nusc is not None:
    
    print(f"\nGenerating debug animation for point index {pt_idx_to_debug} in sweep {TARGET_LIDAR_SD_TOKEN[:8]}...")
    
    # create_point_debug_bev_animation was updated to use mdetector.config_accessor internally
    html_debug_anim, anim_obj, projected_pixel_from_anim = create_point_debug_bev_animation(
        nusc=nusc,
        target_sweep_sd_token=TARGET_LIDAR_SD_TOKEN,
        mdetector=detector, # Pass the MDetector instance
        current_di_from_detector_logic=processed_target_di,
        pt_idx_in_current_di=pt_idx_to_debug,
        idx_of_current_di_in_lib=idx_of_processed_target_di,
        scene_name=target_scene_rec['name'],
        num_past_sweeps_to_show=5, # Or from config_accessor.get_visualization_params().get('debug_animation').get('num_past_sweeps')
        point_downsample_anim=1,   # Or from config
        interval_ms=100,           # Or from config
        figsize_anim=(8, 8),     # Or from config
        save_path=None
    )

    if projected_pixel_from_anim is not None:
        print(f"ANIMATION HELPER: Projected pixel in historical DI for debug point: {projected_pixel_from_anim}")

    if html_debug_anim:
        display(html_debug_anim)
    elif anim_obj:
        print("Animation object created but not displayed (e.g., if saved directly).")
    else:
        print("Debug animation generation failed.")
else:
    print("Cannot generate debug animation: Prerequisites not met.")


In [ ]:


# %% Cell 9: K3D Visualization (Focusing on Point of Interest)
if RUN_K3D:
    # point_global_to_viz was defined in your original Cell 7 (the one before the enhanced K3D)
    # Let's ensure it's defined here based on pt_idx_to_debug and processed_target_di
    point_global_to_viz: Optional[np.ndarray] = None
    if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
    processed_target_di and processed_target_di.original_points_global_coords is not None:
        if 0 <= pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
            point_global_to_viz = processed_target_di.original_points_global_coords[pt_idx_to_debug]

    if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
    processed_target_di and processed_target_di.original_points_global_coords is not None and \
    detector and idx_of_processed_target_di is not None and point_global_to_viz is not None and \
    config_accessor: # Added config_accessor check

        print(f"\nPreparing K3D visualization focusing on Point Index: {pt_idx_to_debug}")
        
        mdet_label_val = processed_target_di.mdet_labels_for_points[pt_idx_to_debug]
        mdet_label_enum = OcclusionResult(mdet_label_val)

        gt_status_for_viz = "GT Unknown"
        if gt_indices_dict_for_target and gt_indices_dict_for_target.get('error_msg') is None: 
            if pt_idx_to_debug in gt_indices_dict_for_target.get('gt_dynamic_indices', []): gt_status_for_viz = "GT Dynamic"
            elif pt_idx_to_debug in gt_indices_dict_for_target.get('gt_static_indices', []): gt_status_for_viz = "GT Static"
            elif pt_idx_to_debug in gt_indices_dict_for_target.get('unlabeled_indices', []): gt_status_for_viz = "GT Unlabeled"
        
        plot_title = f"P{pt_idx_to_debug} (MDet:{mdet_label_enum.name}, GT:{gt_status_for_viz})"
        
        # --- Use visualize_sweep_k3d for a more standard visualization ---
        # Customize its behavior using k3d_plot_params from config_accessor
        
        # Determine paths for K3D visualizer
        gt_hdf5_for_k3d = None
        if GT_LABELS_HDF5_DIR and target_scene_rec:
            gt_hdf5_for_k3d = os.path.join(GT_LABELS_HDF5_DIR, f"gt_point_labels_{target_scene_rec['name']}.h5")
            if not os.path.exists(gt_hdf5_for_k3d): gt_hdf5_for_k3d = None
            
        mdet_hdf5_for_k3d = None
        if MDET_RESULTS_HDF5_DIR and target_scene_rec:
            mdet_hdf5_for_k3d = os.path.join(MDET_RESULTS_HDF5_DIR, f"mdet_results_{target_scene_rec['name']}.h5")
            if not os.path.exists(mdet_hdf5_for_k3d): mdet_hdf5_for_k3d = None

        if nusc and target_scene_rec and TARGET_LIDAR_SD_TOKEN:
            k3d_plot_params = config_accessor.get_k3d_plot_params()
            k3d_plot = visualize_sweep_k3d(
                nusc=nusc,
                scene_token=target_scene_rec['token'],
                target_sweep_lidar_sd_token=TARGET_LIDAR_SD_TOKEN,
                gt_labels_hdf5_path=gt_hdf5_for_k3d,
                mdet_results_hdf5_path=mdet_hdf5_for_k3d,
                config_accessor=config_accessor, # Pass the accessor
                show_gt_boxes=k3d_plot_params.get('show_gt_boxes', True),
                show_gt_points=k3d_plot_params.get('show_gt_points', True),
                show_mdet_points=k3d_plot_params.get('show_mdet_points', True),
                point_size=k3d_plot_params.get('point_size', 0.05),
                downsample_factor=k3d_plot_params.get('downsample_factor_k3d', 1) # Use a specific key for K3D downsample
            )
            if k3d_plot:
                # Add a specific marker for the point of interest if desired
                color_poi = 0xFF0000 
                if mdet_label_enum == OcclusionResult.OCCLUDING_IMAGE: color_poi = 0x00FF00 
                elif mdet_label_enum == OcclusionResult.OCCLUDED_BY_IMAGE: color_poi = 0xFFA500 
                elif mdet_label_enum == OcclusionResult.UNDETERMINED: color_poi = 0xFFFF00 
                
                k3d_plot += k3d.points(
                    positions=point_global_to_viz.reshape(1,3).astype(np.float32),
                    color=color_poi,
                    point_size=k3d_plot_params.get('poi_k3d_point_size', 0.3), # Configurable POI size
                    shader='3d',
                    name=f'DEBUG_POI_P{pt_idx_to_debug}'
                )
                k3d_plot.camera_auto_fit = False # Turn off auto-fit if setting manually
                k3d_plot.camera = [ # Example camera setting, adjust as needed
                    point_global_to_viz[0] + 10, point_global_to_viz[1] + 10, point_global_to_viz[2] + 5,
                    point_global_to_viz[0], point_global_to_viz[1], point_global_to_viz[2],
                    0,0,1
                ]
                display(k3d_plot)
            else:
                print("Failed to create K3D plot using visualize_sweep_k3d.")
        else:
            print("Missing nusc, target_scene_rec, or TARGET_LIDAR_SD_TOKEN for K3D plot.")
    else:
        print("Cannot visualize K3D: Prerequisites not met.")


In [ ]:


# %% Cell 10: Map Consistency Check Investigation

point_global_for_mcc_viz: Optional[np.ndarray] = None # Renamed to avoid conflict
if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   processed_target_di and processed_target_di.original_points_global_coords is not None:
    if 0 <= pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
        point_global_for_mcc_viz = processed_target_di.original_points_global_coords[pt_idx_to_debug]

if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   detector is not None and processed_target_di is not None and \
   point_global_for_mcc_viz is not None:

    print(f"\n\n--- Investigating Map Consistency for P{pt_idx_to_debug} (Current Label in DI {idx_of_processed_target_di}: {OcclusionResult(processed_target_di.mdet_labels_for_points[pt_idx_to_debug]).name}) ---")
    print(f"    Using MDetector.is_map_consistent with actual config values (via accessor).")

    try:
        if not hasattr(detector, 'is_map_consistent'):
            print("  Error: MDetector instance does not have an 'is_map_consistent' method.")
        else:
            current_di_timestamp = processed_target_di.timestamp
            print(f"  Calling detector.is_map_consistent for point P{pt_idx_to_debug} (global coords: {np.round(point_global_for_mcc_viz,2)})")
            print(f"  Timestamp of DI containing P{pt_idx_to_debug}: {current_di_timestamp:.2f}s")
            print(f"  Checking direction: 'past'")

            # is_map_consistent uses detector.config_accessor internally
            is_consistent_with_static_map, mcc_debug_data = detector.is_map_consistent(
                point_global=point_global_for_mcc_viz,
                current_timestamp=current_di_timestamp,
                check_direction='past',
                return_debug_info=True
            )
            print(f"\n  Result of is_map_consistent: {is_consistent_with_static_map}")
            print(f"    (True means the point's location showed consistency with static map elements in the past)")
            
            print(f"\n  Detailed Debug Information from is_map_consistent:")
            # (make_serializable helper and json.dumps remain the same)
            def make_serializable(obj):
                if isinstance(obj, np.ndarray): return obj.tolist()
                if isinstance(obj, OcclusionResult): return obj.name
                if isinstance(obj, tuple): return [make_serializable(item) for item in obj]
                if isinstance(obj, dict): return {k: make_serializable(v) for k, v in obj.items()}
                if isinstance(obj, list): return [make_serializable(i) for i in obj]
                try: 
                    if hasattr(obj, 'item'): return obj.item()
                except Exception: pass
                return obj
            mcc_debug_data_serializable = make_serializable(mcc_debug_data)
            try:
                print(json.dumps(mcc_debug_data_serializable, indent=2))
            except TypeError as e:
                print(f"Error serializing debug data to JSON: {e}. Raw data parts:")
                for key, value in mcc_debug_data.items(): print(f"  {key}: {str(value)[:200]}") # Print truncated value

            # (Interpretation logic remains the same)
            current_label_of_debug_pt = OcclusionResult(processed_target_di.mdet_labels_for_points[pt_idx_to_debug])
            print(f"\n  Interpretation for P{pt_idx_to_debug} (Current Label: {current_label_of_debug_pt.name}):")
            if current_label_of_debug_pt == OcclusionResult.OCCLUDING_IMAGE:
                if is_consistent_with_static_map: 
                    print(f"  ---> DECISION: P{pt_idx_to_debug} IS map consistent. Label likely OVERRIDDEN by MCC.")
                else: 
                    print(f"  ---> DECISION: P{pt_idx_to_debug} is NOT map consistent. Label likely RETAINED by MCC.")
            # ... (other interpretations) ...
    except Exception as e:
        print(f"  An error occurred during map consistency investigation: {e}")
        import traceback
        print(traceback.format_exc())
else:
    print("Cannot investigate map consistency: Prerequisites not met.")



In [ ]:

# %% Cell 11: Direct NuScenes GT Check

# point_global_for_direct_gt_check: Optional[np.ndarray] = None # Renamed
# if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
#    processed_target_di and processed_target_di.original_points_global_coords is not None:
#     if 0 <= pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
#         point_global_for_direct_gt_check = processed_target_di.original_points_global_coords[pt_idx_to_debug]

# Ensure velocity_threshold_gt is available from Cell 4/6
if 'velocity_threshold_gt' not in locals() or velocity_threshold_gt is None:
    if config_accessor:
        velocity_threshold_gt = config_accessor.get_validation_params().get('gt_velocity_threshold', 0.5)
    else: velocity_threshold_gt = 0.5


if nusc is not None and TARGET_LIDAR_SD_TOKEN is not None and \
   'point_global_for_mcc_viz' in locals() and point_global_for_mcc_viz is not None and \
   velocity_threshold_gt is not None: # Use the same point_global as MCC check

    print(f"\n\n--- Direct NuScenes GT Check for P{pt_idx_to_debug} in Keyframe {TARGET_LIDAR_SD_TOKEN[:8]} ---")
    try:
        lidar_sweep_rec = nusc.get('sample_data', TARGET_LIDAR_SD_TOKEN)
        sample_token = lidar_sweep_rec['sample_token']
        sample_rec = nusc.get('sample', sample_token)
        print(f"  Target Sweep is Keyframe: {lidar_sweep_rec['is_key_frame']}")
        if not lidar_sweep_rec['is_key_frame']:
            print("  WARNING: Target sweep is NOT a keyframe. GT annotations might be less direct.")

        print(f"  P{pt_idx_to_debug} Global Coords (from M-Detector): {np.round(point_global_for_mcc_viz, 3)}")
        found_in_gt_box = False
        for ann_token in sample_rec['anns']:
            ann_rec = nusc.get('sample_annotation', ann_token)
            box = NuScenesDataClassesBox(ann_rec['translation'], ann_rec['size'], Quaternion(ann_rec['rotation']),
                                         name=ann_rec['category_name'], token=ann_rec['token'])
            point_in_box_frame = np.array(point_global_for_mcc_viz) - box.center
            point_in_box_frame = box.orientation.inverse.rotate(point_in_box_frame)
            half_width, half_length, half_height = box.wlh / 2.0
            is_inside = (abs(point_in_box_frame[0]) <= half_length and \
                         abs(point_in_box_frame[1]) <= half_width  and \
                         abs(point_in_box_frame[2]) <= half_height)
            if is_inside:
                found_in_gt_box = True
                print(f"\n  ---> P{pt_idx_to_debug} FOUND inside GT Annotation:")
                # ... (rest of the print statements for GT box info, velocity, etc. remain the same) ...
                # ... (ensure velocity_threshold_gt is used for dynamic/static assessment) ...
                print(f"    Annotation Token: {ann_token}")
                print(f"    Instance Token: {ann_rec['instance_token']}")
                print(f"    Category: {ann_rec['category_name']}")
                try:
                    gt_box_velocity_global = nusc.box_velocity(ann_token)
                    if np.any(np.isnan(gt_box_velocity_global)):
                        print("    GT Box Velocity (SDK): NaN")
                    else:
                        gt_box_speed = np.linalg.norm(gt_box_velocity_global[:2])
                        print(f"    GT Box Velocity (SDK, Global): {np.round(gt_box_velocity_global, 2)}")
                        print(f"    GT Box Speed (SDK, Horizontal): {gt_box_speed:.2f} m/s")
                        if gt_box_speed < velocity_threshold_gt: print(f"    ---> SDK: GT Box STATIC")
                        else: print(f"    ---> SDK: GT Box DYNAMIC")
                except Exception as e_vel: print(f"    Error getting GT Box Velocity: {e_vel}")
                break
        if not found_in_gt_box:
            print(f"  P{pt_idx_to_debug} was NOT found inside any GT annotation box for this keyframe sample.")
    except Exception as e:
        print(f"  An error occurred during Direct NuScenes GT Check: {e}")
else:
    print("Cannot perform Direct NuScenes GT Check: Prerequisites not met.")



In [ ]:

# %% Cell 12: Raw Occlusion History

if detector and processed_target_di and \
   hasattr(processed_target_di, 'raw_occlusion_results_vs_history') and \
   processed_target_di.raw_occlusion_results_vs_history is not None and \
   'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   idx_of_processed_target_di is not None and \
   config_accessor: # Added config_accessor

    print(f"\n--- Raw Occlusion History for P{pt_idx_to_debug} (from current_di.raw_occlusion_results_vs_history) ---")
    if processed_target_di.raw_occlusion_results_vs_history.shape[0] > pt_idx_to_debug:
        # --- MODIFIED: Get N from M-Detector's config_accessor for Test1 (Event Test) ---
        # Assuming Test1 parameters are what you mean by "Event Test" here
        test1_params = config_accessor.get_test1_perpendicular_params() # Or get_event_detection_logic_params().get('test1_perpendicular')
        num_hist_d_for_event_test = test1_params.get('num_historical_DIs_N', 5)
        # --- End MODIFIED ---
        
        actual_history_len = min(num_hist_d_for_event_test, processed_target_di.raw_occlusion_results_vs_history.shape[1])
        history_slice = processed_target_di.raw_occlusion_results_vs_history[pt_idx_to_debug, :actual_history_len]
        
        print(f"  P{pt_idx_to_debug} (current DI {idx_of_processed_target_di}) vs. N={actual_history_len} historical DIs:")
        for k_hist, result_val in enumerate(history_slice):
            result_enum = OcclusionResult(result_val)
            hist_di_for_ts_idx = idx_of_processed_target_di - 1 - k_hist
            hist_di_for_ts = detector.depth_image_library.get_image_by_index(hist_di_for_ts_idx)
            hist_ts_str = f"(DI Idx: {hist_di_for_ts_idx}, TS: {hist_di_for_ts.timestamp:.2f})" if hist_di_for_ts else f"(DI Idx: {hist_di_for_ts_idx} not found)"
            interp_str = ""
            if result_enum == OcclusionResult.OCCLUDED_BY_IMAGE: interp_str = f"-> P{pt_idx_to_debug}'s loc was BEHIND pts"
            elif result_enum == OcclusionResult.OCCLUDING_IMAGE: interp_str = f"-> P{pt_idx_to_debug}'s loc was IN FRONT OF pts"
            elif result_enum == OcclusionResult.EMPTY_IN_IMAGE: interp_str = f"-> P{pt_idx_to_debug}'s loc projected to EMPTY region"
            print(f"    vs. Hist DI-{k_hist} {hist_ts_str}: {result_enum.name} {interp_str}")
    else:
        print(f"  raw_occlusion_results_vs_history not available or pt_idx {pt_idx_to_debug} out of bounds.")
else:
    print("Cannot print raw occlusion history: Prerequisites not met.")



In [ ]:

# %% Cell 13: Enhanced K3D Plot with History and GT Boxes
if RUN_K3D:
    # Ensure point_global_to_viz is defined from a previous cell (e.g., Cell 9 or Cell 6)
    if 'point_global_to_viz' not in locals(): point_global_to_viz = None 
    if 'initial_cam' not in locals(): initial_cam = None # From original Cell 7 K3D plot

    if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
    detector and processed_target_di and idx_of_processed_target_di is not None and \
    point_global_to_viz is not None and nusc and config_accessor: # Added config_accessor

        print(f"\n\n--- Enhanced K3D Visualization for P{pt_idx_to_debug} and its History ---")
        points_to_plot_k3d_enhanced = {}
        current_di_points_global = processed_target_di.get_original_points_global()

        # --- Use k3d_plot_params from config_accessor for styling ---
        k3d_plot_params = config_accessor.get_k3d_plot_params()
        poi_color = k3d_plot_params.get('poi_color_hex', 0xFF0000) # Example default
        poi_size = k3d_plot_params.get('poi_k3d_point_size', 1.0)
        current_di_color = k3d_plot_params.get('current_di_context_color_hex', 0xCCCCCC)
        current_di_size = k3d_plot_params.get('current_di_context_point_size', 0.1)
        # Define more colors/sizes in your k3d_plot_params for historical DIs if needed
        # ---

        if point_global_to_viz is not None:
            points_to_plot_k3d_enhanced[f'POI_P{pt_idx_to_debug}_Current'] = (point_global_to_viz.reshape(1, 3), poi_color, poi_size)
        if current_di_points_global is not None:
            points_to_plot_k3d_enhanced['Current_DI_AllPts'] = (current_di_points_global, current_di_color, current_di_size)

        # (Loop for historical DIs and context region points remains the same,
        #  but you can use k3d_plot_params for their colors/sizes too)
        # Example for immediate past:
        idx_imm_past = idx_of_processed_target_di - 1
        imm_past_di = detector.depth_image_library.get_image_by_index(idx_imm_past)
        if imm_past_di and imm_past_di.get_original_points_global() is not None:
            points_to_plot_k3d_enhanced[f'Past_DI_Idx{idx_imm_past}_AllPts'] = (
                imm_past_di.get_original_points_global(), 
                k3d_plot_params.get('hist_di_1_color_hex', 0x00FF00), # Green
                k3d_plot_params.get('hist_di_1_point_size', 0.1)
            )
            # ... (context region highlighting) ...

        # (Loop for other past DIs remains the same, use k3d_plot_params for styling)

        plot_title_enhanced = f"Enhanced K3D for P{pt_idx_to_debug} & History (Current DI {idx_of_processed_target_di})"
        if initial_cam is None and point_global_to_viz is not None: # Define a default camera if not set
            initial_cam = [
                point_global_to_viz[0] + 15, point_global_to_viz[1] + 15, point_global_to_viz[2] + 5,
                point_global_to_viz[0], point_global_to_viz[1], point_global_to_viz[2], 0,0,1
            ]

        enhanced_plot = create_k3d_plot_with_points(
            points_to_plot_k3d_enhanced,
            plot_title=plot_title_enhanced,
            camera_auto_fit=False if initial_cam else True,
            initial_camera_settings=initial_cam,
            background_color=k3d_plot_params.get('plot_background_color_hex', 0xAAAAAA)
        )

        # --- Adding 3D GT Annotation Boxes (using k3d_plot_params from accessor) ---
        if enhanced_plot and TARGET_LIDAR_SD_TOKEN: # TARGET_LIDAR_SD_TOKEN check was missing
            try:
                lidar_sd_rec_for_scene = nusc.get('sample_data', TARGET_LIDAR_SD_TOKEN)
                sample_rec_for_scene = nusc.get('sample', lidar_sd_rec_for_scene['sample_token'])
                scene_token_for_boxes = sample_rec_for_scene['scene_token']
                
                instance_tokens_in_scene = find_instances_in_scene(nusc, scene_token_for_boxes, min_annotations=1)
                all_scene_sweeps_list = list(get_scene_sweep_data_sequence(nusc, scene_token_for_boxes))
                current_sweep_idx_in_list = -1
                for idx, sdd_box_loop_var in enumerate(all_scene_sweeps_list):
                    if sdd_box_loop_var['lidar_sd_token'] == TARGET_LIDAR_SD_TOKEN:
                        current_sweep_idx_in_list = idx; break
                
                if current_sweep_idx_in_list != -1 and instance_tokens_in_scene:
                    print(f"  Adding 3D GT boxes to K3D plot...")
                    num_instance_colors = k3d_plot_params.get('num_instance_box_colors', 20)
                    instance_colormap_name = k3d_plot_params.get('instance_box_colormap', 'tab20')
                    color_map_instances = plt.cm.get_cmap(instance_colormap_name, num_instance_colors)
                    inst_color_idx = 0
                    for inst_token in instance_tokens_in_scene:
                        boxes_for_inst, _, _, _ = get_interpolated_extrapolated_boxes_for_instance(nusc, inst_token, all_scene_sweeps_list)
                        gt_box_at_sweep: Optional[NuScenesDataClassesBox] = boxes_for_inst[current_sweep_idx_in_list]
                        if gt_box_at_sweep:
                            # ... (box drawing logic remains the same, using k3d_plot_params.get('gt_box_line_width', 0.03))
                            instance_color_rgb = color_map_instances(inst_color_idx % num_instance_colors)[:3]
                            instance_color_hex = int(instance_color_rgb[0]*255)<<16 | int(instance_color_rgb[1]*255)<<8 | int(instance_color_rgb[2]*255)
                            inst_color_idx += 1
                            corners = gt_box_at_sweep.corners() 
                            box_edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
                            for start_idx_edge, end_idx_edge in box_edges: # Renamed to avoid conflict
                                segment_vertices = corners[:, [start_idx_edge, end_idx_edge]].T.astype(np.float32)
                                enhanced_plot += k3d.line(segment_vertices, shader='simple', color=instance_color_hex,
                                                        width=k3d_plot_params.get('gt_box_line_width', 0.03),
                                                        name=f'GTBox3D_{inst_token[:6]}_{gt_box_at_sweep.name[:5]}')
                    print(f"  Added 3D GT boxes.")
            except Exception as e_box: print(f"  Error adding 3D GT boxes: {e_box}")
        # --- End Adding GT Boxes ---

        if enhanced_plot:
            display(enhanced_plot)
        else:
            print("Failed to create enhanced K3D plot.")
    else:
        print("Cannot create enhanced K3D plot: Prerequisites not met.")



In [ ]:

# %% Cell 14: Camera View with Annotations
# (This cell primarily uses nusc and matplotlib, less dependent on MDet config directly,
#  but ensure TARGET_LIDAR_SD_TOKEN is correctly set from Cell 1)

if nusc and TARGET_LIDAR_SD_TOKEN:
    print(f"\n\n--- Camera View for Context (CAM_FRONT_LEFT) with Annotations ---")
    try:
        lidar_sweep_rec = nusc.get('sample_data', TARGET_LIDAR_SD_TOKEN)
        sample_token = lidar_sweep_rec['sample_token']
        sample_rec = nusc.get('sample', sample_token)
        cam_front_left_token = sample_rec['data'].get('CAM_FRONT_LEFT')
        if cam_front_left_token:
            print(f"  Displaying CAM_FRONT_LEFT image for sample_token: {sample_token}")
            fig_cam_ann, ax_cam_ann = plt.subplots(1, 1, figsize=(12, 9))
            nusc.render_sample_data(cam_front_left_token, with_anns=True, verbose=False, ax=ax_cam_ann)
            ax_cam_ann.set_title(f"CAM_FRONT_LEFT (with Anns) - Sweep: {TARGET_LIDAR_SD_TOKEN[:8]}")
            plt.show()
        else: print(f"  CAM_FRONT_LEFT data not found for sample_token: {sample_token}")
    except Exception as e: print(f"  Error displaying camera image: {e}")
else:
    print("Cannot display camera view: NuScenes SDK not ready or TARGET_LIDAR_SD_TOKEN not set.")

print("\nNotebook Execution Finished.")

In [ ]:
event_tests_logger = logging.getLogger('src.core.m_detector.event_tests')
event_tests_logger.setLevel(logging.DEBUG) 
# You might need to add a handler if it doesn't output to notebook console
# stream_handler = logging.StreamHandler(sys.stdout) # Example handler
# if not event_tests_logger.hasHandlers(): event_tests_logger.addHandler(stream_handler)

print("Logging for event_tests set to DEBUG.")

In [ ]:
# %% Cell 10b: Deep Dive into Test 2 (Parallel Motion) for P{pt_idx_to_debug}

# --- Ensure point_global_to_viz is defined for the point of interest ---
point_global_to_viz: Optional[np.ndarray] = None 
if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   'processed_target_di' in locals() and processed_target_di is not None and \
   processed_target_di.original_points_global_coords is not None:
    if 0 <= pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
        point_global_to_viz = processed_target_di.original_points_global_coords[pt_idx_to_debug]
    else:
        print(f"Warning in Cell 10b: pt_idx_to_debug ({pt_idx_to_debug}) is out of bounds for processed_target_di.original_points_global_coords.")
# --- End ensure point_global_to_viz ---

if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   detector is not None and processed_target_di is not None and \
   idx_of_processed_target_di is not None and \
   point_global_to_viz is not None: # Now check if point_global_to_viz was successfully defined

    print(f"\n\n--- Investigating Test 2 (Parallel Motion) for P{pt_idx_to_debug} ---")
    print(f"    (P{pt_idx_to_debug} global coords: {np.round(point_global_to_viz,2)})")
    print(f"    (Current DI TS: {processed_target_di.timestamp:.2f}, Lib Idx: {idx_of_processed_target_di})")
    print(f"    (M2 from config: {detector.test2_M2_depth_images})")

    # Ensure the event_tests logger is set to DEBUG if you want to see internal logs
    # logging.getLogger('src.core.m_detector.event_tests').setLevel(logging.DEBUG)

    test2_passed, test2_debug_details = detector.execute_test2_parallel_motion(
        point_global_P=point_global_to_viz,
        current_di_timestamp=processed_target_di.timestamp,
        current_di_idx_in_lib=idx_of_processed_target_di,
        return_debug_info=True
    )

    print(f"\n  Test 2 Overall Result for P{pt_idx_to_debug}: {'PASSED' if test2_passed else 'FAILED'}")

    if test2_debug_details:
        print(f"\n  Detailed Chain for Test 2 (P{pt_idx_to_debug}):")
        for i, step_info in enumerate(test2_debug_details):
            print(f"  -----------------------------------------------------")
            print(f"  Step {step_info.get('step', i)}: {step_info.get('status', 'N/A')}")
            # Ensure timestamp formatting handles None gracefully or checks before formatting
            origin_di_ts_pto = step_info.get('origin_di_timestamp_of_pto')
            hist_di_ts = step_info.get('historical_di_checked_timestamp')

            print(f"    Origin DI TS of p_to_be_occluded: {f'{origin_di_ts_pto:.2f}' if origin_di_ts_pto is not None else 'N/A'}")

            if step_info.get('historical_di_checked_idx_in_lib') is not None:
                print(f"    Checked against Hist. DI Lib Idx: {step_info.get('historical_di_checked_idx_in_lib')}, TS: {f'{hist_di_ts:.2f}' if hist_di_ts is not None else 'N/A'}")
            if step_info.get('projection_in_hist_successful'):
                print(f"    Projected to pixel in Hist DI: {step_info.get('projected_pixel_in_hist')}")
                print(f"    Num candidate occluders in pixel: {step_info.get('num_candidates_in_projected_pixel', 'N/A')}")
            
            occlusion_checks = step_info.get('occlusion_check_details', [])
            if occlusion_checks:
                print(f"    Occlusion checks on candidates in Hist DI:")
                for cand_idx, check in enumerate(occlusion_checks):
                    mcc_ts = check.get('mcc_timestamp')
                    print(f"      Cand {cand_idx}: Orig Idx {check.get('hist_candidate_original_idx')}, Coords: {check.get('hist_candidate_global')}")
                    print(f"        Detailed Occlusion (P occluded by Cand): {check.get('detailed_occlusion_result')}")
                    if check.get('mcc_applied_on_point'):
                        print(f"        MCC on: {check.get('mcc_applied_on_point')} (Coords: {check.get('mcc_point_coords')}, TS: {f'{mcc_ts:.2f}' if mcc_ts is not None else 'N/A'})")
                        print(f"        MCC Result (Is Static): {check.get('mcc_result_is_consistent_with_map')}")
                        print(f"        MCC Rejects this Link: {check.get('mcc_rejects_this_link')}")
                    print(f"        Forms Next Link in Chain: {check.get('forms_next_link_in_chain')}")
            if step_info.get("status", "").startswith("Success") and step_info.get("selected_occluder_for_next_step"):
                 print(f"    Selected occluder for next step: {step_info.get('selected_occluder_for_next_step')}")
        print(f"  -----------------------------------------------------")
    else:
        print("  No detailed debug information returned for Test 2.")
else:
    print("Cannot perform Test 2 deep dive: Prerequisites not met (e.g., pt_idx_to_debug, detector, processed_target_di, or point_global_to_viz missing/invalid).")

In [ ]:
# %% Cell 10c: Deep Dive into Test 3 (Perpendicular Revealing) for P{pt_idx_to_debug}

# --- Ensure point_global_to_viz is defined for the point of interest ---
# This might be redundant if Cell 10b already defined it and ran successfully,
# but it's safe to include for robustness if cells are run independently.
if 'point_global_to_viz' not in locals() or point_global_to_viz is None: # Check if it needs re-defining
    point_global_to_viz: Optional[np.ndarray] = None 
    if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
       'processed_target_di' in locals() and processed_target_di is not None and \
       processed_target_di.original_points_global_coords is not None:
        if 0 <= pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
            point_global_to_viz = processed_target_di.original_points_global_coords[pt_idx_to_debug]
        else:
            print(f"Warning in Cell 10c: pt_idx_to_debug ({pt_idx_to_debug}) is out of bounds for processed_target_di.original_points_global_coords.")
# --- End ensure point_global_to_viz ---


if 'pt_idx_to_debug' in locals() and pt_idx_to_debug is not None and \
   detector is not None and processed_target_di is not None and \
   idx_of_processed_target_di is not None and \
   point_global_to_viz is not None: # Check if point_global_to_viz was successfully defined

    print(f"\n\n--- Investigating Test 3 (Perpendicular Revealing) for P{pt_idx_to_debug} ---")
    print(f"    (P{pt_idx_to_debug} global coords: {np.round(point_global_to_viz,2)})")
    print(f"    (Current DI TS: {processed_target_di.timestamp:.2f}, Lib Idx: {idx_of_processed_target_di})")
    print(f"    (M3 from config: {detector.test3_M3_depth_images})")

    test3_passed, test3_debug_details = detector.execute_test3_perpendicular_motion(
        point_global_P=point_global_to_viz,
        current_di_timestamp=processed_target_di.timestamp,
        current_di_idx_in_lib=idx_of_processed_target_di,
        return_debug_info=True
    )

    print(f"\n  Test 3 Overall Result for P{pt_idx_to_debug}: {'PASSED' if test3_passed else 'FAILED'}")

    if test3_debug_details:
        print(f"\n  Detailed Chain for Test 3 (P{pt_idx_to_debug}):")
        for i, step_info in enumerate(test3_debug_details):
            print(f"  -----------------------------------------------------")
            print(f"  Step {step_info.get('step', i)}: {step_info.get('status', 'N/A')}")
            
            mcc_P_info_step = step_info.get("mcc_details_on_P")
            initial_mcc_P_info_step0 = step_info.get("initial_mcc_on_point_P_details") # For step 0
            hist_di_ts_t3 = step_info.get('historical_di_checked_timestamp')


            if step_info.get('step', -2) == -1 :
                if mcc_P_info_step:
                    mcc_ts_p = mcc_P_info_step.get('mcc_timestamp')
                    print(f"    Initial MCC on P{pt_idx_to_debug} (Coords: {mcc_P_info_step.get('mcc_point_coords')}, TS: {f'{mcc_ts_p:.2f}' if mcc_ts_p is not None else 'N/A'})")
                    print(f"    MCC Result (P is Static): {mcc_P_info_step.get('mcc_result_is_consistent_with_map')}")
                continue

            hist_di_ts_t3 = step_info.get('historical_di_checked_timestamp')
            print(f"    Point that occludes: {step_info.get('point_that_occludes_global', 'N/A')}")
            if step_info.get('historical_di_checked_idx_in_lib') is not None:
                    print(f"    Checked against Hist. DI Lib Idx: {step_info.get('historical_di_checked_idx_in_lib')}, TS: {f'{hist_di_ts_t3:.2f}' if hist_di_ts_t3 is not None else 'N/A'}")
            if step_info.get('projection_in_hist_successful'):
                print(f"    Projected to pixel in Hist DI: {step_info.get('projected_pixel_in_hist')}")
                print(f"    Num candidate points to be occluded in pixel: {step_info.get('num_candidates_in_projected_pixel', 'N/A')}")
            
            if step_info.get('step') == 0 and initial_mcc_P_info_step0:
                print(f"    (Initial MCC on P{pt_idx_to_debug} was: {'STATIC' if initial_mcc_P_info_step0.get('mcc_result_is_consistent_with_map') else 'NOT STATIC (good for Test 3)'})")

            occlusion_checks = step_info.get('occlusion_check_details', [])
            if occlusion_checks:
                print(f"    Occlusion checks on candidates in Hist DI:")
                for cand_idx, check in enumerate(occlusion_checks):
                    print(f"      Cand {cand_idx}: Orig Idx {check.get('hist_candidate_original_idx')}, Coords: {check.get('hist_candidate_being_occluded_global')}")
                    print(f"        Detailed Occlusion (Chain Point occludes Cand): {check.get('detailed_occlusion_result')}")
                    print(f"        Forms Next Link in Chain: {check.get('forms_next_link_in_chain')}")
            if step_info.get("status", "").startswith("Success") and step_info.get("selected_occluded_point_for_next_step"):
                 print(f"    Selected occluded point (becomes occluder for next step): {step_info.get('selected_occluded_point_for_next_step')}")
        print(f"  -----------------------------------------------------")
    else:
        print("  No detailed debug information returned for Test 3.")
else:
    print("Cannot perform Test 3 deep dive: Prerequisites not met (e.g., pt_idx_to_debug, detector, processed_target_di, or point_global_to_viz missing/invalid).")